In [22]:
import torch
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct
from qdrant_client.models import Distance, VectorParams

In [23]:
client = QdrantClient(url="http://localhost:6333")

In [24]:
client.create_collection(
    collection_name="test_collection",
    vectors_config=VectorParams(size=1024, distance=Distance.DOT),
)

True

In [19]:
random_embeddings = np.random.normal(size=(10000, 1024))
random_embeddings /= np.sqrt((random_embeddings**2).sum(axis=-1))[:, None]

In [21]:
all_points = [PointStruct(id=l, vector = embed, payload={"city": "Berlin"}) for l, embed in enumerate(random_embeddings)]

In [22]:
operation_info = client.upsert(
    collection_name="test_collection",
    wait=True,
    points=all_points[:1000],
)


In [27]:
query = np.random.normal(size=(1024,))
query = query/np.sqrt((query**2).sum())

In [28]:
search_result = client.query_points(
    collection_name="test_collection",
    query=query,
    with_payload=False,
    limit=3
).points

In [29]:
search_result

[ScoredPoint(id=845, version=2, score=0.10005581, payload=None, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=791, version=2, score=0.08682226, payload=None, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=810, version=2, score=0.086214915, payload=None, vector=None, shard_key=None, order_value=None)]

In [40]:
np.argmax(random_embeddings[:1000]@query)

np.int64(845)

## Trying my vector db

In [25]:
import pandas as pd
from datasets import Dataset, load_dataset
from sentence_transformers import SentenceTransformer

In [26]:
chunks = load_dataset("parquet", data_files = "/proj/berzelius-2025-303/users/x_gabdu/random/ProjectAlbalat/albalat/data/processed/chunks.parquet")["train"]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

In [27]:
client = QdrantClient(url="http://localhost:6333")

In [28]:
test_query = np.random.normal(size=1024)

In [29]:
search_result = client.query_points(
    collection_name="v1_m_16_efConstruct_100",
    query=test_query,
    with_payload=False,
    limit=10
).points

print(search_result)

[ScoredPoint(id=8355703, version=229, score=0.13570404, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=10688926, version=12929, score=0.13521576, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=2899558, version=9903, score=0.13403702, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=14342010, version=5154, score=0.13383865, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=8922381, version=8248, score=0.13149643, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=8468502, version=2908, score=0.13128662, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=10836735, version=9532, score=0.13119888, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=2928459, version=9799, score=0.13081741, payload=None, vector=None, shard_key=None, order_value=None), ScoredPoint(id=2391536, version=9472, score=0.128768

In [30]:
model_bf16 = SentenceTransformer("IEITYuan/Yuan-embedding-2.0-en", device="cpu", 
                            model_kwargs={"dtype": torch.bfloat16, "attn_implementation": "sdpa"})

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [31]:
def query_vector_index(query: str, top_k: int = 100):# -> Dataset:
    query_vector = model_bf16.encode(query, normalize_embeddings=True).astype(np.float16)
    search_result = client.query_points(
    collection_name="v1_m_16_efConstruct_100",
    query=query_vector,
    with_payload=False,
    limit=top_k
    ).points
    print(mapping_index_id[0])
    all_ids = [mapping_index_id[p.id] for p in search_result]
    all_id_scores = {mapping_index_id[p.id]: p.score for p in search_result}
    relevant_results = chunks.select(all_ids)
    #sorted_p = sorted([(all_id_scores[mapping_index_id[p["index"]]], p["index"], p["text_ids"], p["paragraphs"]) for p in relevant_results], reverse=True)
    sorted_p = [(all_id_scores[mapping_index_id[p["index"]]], p["index"], p["text_ids"], p["paragraphs"]) for p in relevant_results]
    df = pd.DataFrame.from_records(sorted_p)
    df.columns = ["scores", "ids", "text_ids", "paragraphs"]
    return df

def apply_cutoff_scores(df_results: pd.DataFrame, threshold: float) -> pd.DataFrame:
    return df_results[df_results.scores > threshold]


def query_endpoint(query: str, threshold: float, top_k: int = 100) -> pd.DataFrame:
    df_results = query_vector_index(query, top_k)
    df_threshold = apply_cutoff_scores(df_results, threshold)
    return df_threshold
    

In [9]:
#from tqdm import tqdm
#mapping_index_id = {p["index"]:i for i, p in tqdm(enumerate(chunks))}

In [32]:
#np.save("mapping.npy", mapping_index_id)
mapping_index_id = np.load("mapping.npy", allow_pickle=True).item()

In [33]:
for k, v in mapping_index_id.items():
    print(k, v)
    break

3793654 0


In [34]:
query_text = "It was a evening of July. The sun slowly went down while the sky adorned itself with the most beautiful pink colors."
query_text2 = """It was about seven o'clock of an evening in late summer, and across that bleak, barren bit of land the sun was just setting. 
                As they drove along, it sparkled on the window panes of the houses and lit up the cross on the Catholic church; beyond the village 
                it seemed to confine itself to the rocks by the wayside. It turned them a dull soft gold. A strong salt breeze was blowing."""

query_text3 = "The storm was powerful and the waves humonguous. The ship threatened to collapse at any moment."
query_text4 = "I was standing here, among the dead bodies. The morgue felt cold, and I was cold inside, looking at my father lying on a table."
query_text5 = "The plane glided in the air. Its two gigantic wings were reflecting the sunlight while its engine was roaring in the sky."

In [35]:
df_results = query_vector_index(query_text5, 1000)
df_thresholded = apply_cutoff_scores(df_results, 0.6)

255968


In [36]:
eval_dataset = [query_text, query_text2, query_text3, query_text4, query_text5]

In [37]:
import os
import ragas
import asyncio
from openai import AsyncOpenAI
from ragas.metrics import ContextRelevance
#from ragas_examples.improve_rag.rag import RAG, BM25Retriever

In [38]:
#os.environ["OPENAI_API_KEY"] = openai_api_key

In [39]:
openai_client = AsyncOpenAI()

# Create retriever and RAG system

# Query the system
question = "What architecture is the `tokenizers-linux-x64-musl` binary designed for?"
result = query_endpoint(question, 0.8)
print(f"Answer: {result['answer']}")

255968


KeyError: 'answer'

In [40]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision

In [86]:
from pydantic import BaseModel, Field
from collections.abc import Iterable
from langchain_core.prompt_values import StringPromptValue
client = AsyncOpenAI()
evaluator_llm = llm_factory("gpt-4o-mini", client=client)


class Score(BaseModel):
    """
    Pydantic class to check that the data grading of the LLM follows the right format
    """
    rating: int = Field(ge=0, le=2)


class OpenAIScorer():
    """
    Uses the OpenAI API to score the retrieved chunks against the query chunks.
    We use the same LLM with two different prompts, for robustness, following RAGAS.
    """
    def __init__(self, client):
        super(OpenAIScorer, self).__init__()
        """
        llm is an evaluator llm from llm_factory in RAGAS. 
        """
        self.client = client
        self.prompt1 = """You are a world class literary expert designed to evaluate the narrative and semantic
                                similarity between two passages.\nYour task is to determine if the Context is narratively and sementically similar to the Passage.
                                \nDo not rely on your previous knowledge about the Passage.\nUse only what is written in the Context 
                                and in the Passage.\nFollow the instructions below:\n0. If the Context is not narratively and sementically similar to the Passage or it is irrelevant or it is not a literary passage at all, say 0.\n1. 
                                If the Context is partially narratively and sementically similar to the the Passage, 
                                say 1.\n2. If the Context is narratively and sementically similar to the Passage, say 2.\nYou must ONLY provide the 
                                relevance score of 0, 1, or 2, nothing else.\nDo not explain.\nReturn your response as JSON in this format: {"rating": X}
                                where X is 0, 1, or 2. \n
                                Evaluate now.
                                """

        self.prompt2 = """As a specially designed expert to assess the relevance score of a given Passage in the relation to Context, my task is to determine the extent to which the Context is
                         narratively and semantically similar to the Passage. I will rely solely on the information provided in the Passage and Context, and not on any prior knowledge.\n\n
                         Here are the instructions I will follow:\n
                         * If the Context is not narratively and semantically similar to the Passage, I will respond with a relevance score of 0.\n
                         * If the Context is partially narratively and semantically similar to the Passage, I will respond with a relevance score of 1.\n
                         * If the Context is narratively and semantically similar to the Passage, I will respond with a relevance score of 2.\n\n
                         I must provide the relevance score of 0, 1, or 2, nothing else.\n I do not explain.\n I return my response as JSON in this format: {"rating": X}
                                where X is 0, 1, or 2.
                       """        

    def create_queries_unique_prompt(self, query, chunks, prompt_id):
        """
        Create the queries corresponding to a unique prompt.
        """
        if prompt_id not in (1, 2):
            raise ValueError(f"""prompt_id must be 1 or 2, currently {prompt_id}""")
            
        if prompt_id == 1:
            prompt = self.prompt1
        else:
            prompt = self.prompt2

        all_openAIQuery = [f"""Passage: {query}\n\nContext:{chunk}\n\n {prompt}""" for chunk in chunks]
        return all_openAIQuery
        
    def create_queries(self, query, chunks):
        """
        Create the queries by inserting the query and the chunks in both prompts.
        """
        queriesJudge1 = self.create_queries_unique_prompt(query, chunks, 1)
        queriesJudge2 = self.create_queries_unique_prompt(query, chunks, 2)
        return queriesJudge1, queriesJudge2  

    async def get_scores(self, queriesJudge, temperature=0.1):
        """
        Actually sends the full prompt to openAI.
        """
        response = await self.client.chat.completions.create(
            model="gpt-4o-mini",
                  response_format={"type": "json_object"},
                  messages=[{
                            "role": "system",
                            "content": "You are a literary relevance evaluator. Output only JSON."
                        },
                    {"role": "user", "content": queriesJudge}],
            temperature=temperature,
        )
        json_grade = Score.model_validate_json(response.choices[0].message.content)
        return json_grade

    def average_scores(self, scores: tuple[str, str])-> dict[str, float]:
        """
        Average the scores of all the judges for a single context and keeping them.
        """
        average = sum(score.rating for score in scores)/len(scores)
        grades = {f"judge{i}":score.rating for i, score in enumerate(scores)}
        grades.update({"average":average})
        return grades
        

    def process_responses(self, score_judges_1_2: Iterable[tuple[str, str]])-> list[float]:
        """
        Average the scores for each context given by the two judges
        """
        average_scores = [self.average_scores(scores_1_2) for scores_1_2 in score_judges_1_2]
        return average_scores
            
    async def query_openai(self, query: str, chunks: list[str])-> list[float]:
        """
        Queries the self.llm from OpenAI API, with all the pairs (query, chunks), with two prompts each.
        """
        queriesJudge1, queriesJudge2 = self.create_queries(query, chunks)
        scores1, scores2 = await asyncio.gather(
    asyncio.gather(*(self.get_scores(q) for q in queriesJudge1)),
    asyncio.gather(*(self.get_scores(q) for q in queriesJudge2)),
        )
        averaged_scores = self.process_responses(zip(scores1, scores2))
        return averaged_scores

    


In [87]:
a = ("a", "b")
type(a[0])
e = """{"rating": 1}"""

In [88]:
import json
json.loads(e)

{'rating': 1}

In [89]:
'rating' in eval(e).keys()

True

In [90]:
scorer = OpenAIScorer(client)

In [91]:
chunks_retrieved = ['The passengers in the huge plane high above them gave little thought to what passed below, engrossed with their papers or books, or engaged in casual conversation. This monotonous trip was boring to most of them. It seemed a waste of time to spend six good hours in a short 3,500 mile trip. There was nothing to do, nothing to see, except a slowly passing landscape ten miles below. No details could be distinguished, and the steady low throb of the engines, the whirring of the giant propellers, the muffled roar of the air, as it rushed by, combined to form a soothing lullaby of power. It was all right for pleasure seekers and vacationists, but business men were in a hurry.',
                                                                                                                                                                                                                                                                                                         'The boys could now hear the whirring of the motor. Fifty yards away the aëroplane began to descend. Gracefully it volplaned to the earth under perfect control. It landed safely, rolled a little way, and stopped.\n\nThe boys, without a second thought, raced down the slope to greet the aviator, like one of their own kind should be greeted, but as quickly halted as they drew nearer.',
                                                                                                                                                                                                                                        'They flew over the trees, eastward to the prairie land, and then on through the coastal plain to the Atlantic Ocean. Whether they were crossing Florida or Georgia, Linda did not know, and for once she was not interested in the country. The sun rose as they came to the water, but that beautiful sight, too, made no impression upon the unhappy girl. Nothing but the sight of a plane or a boat--the promise of rescue--could have any meaning for her.', 
                   "The house was burning in fire, when I saw my brother break out from the window, followed by flames."]

In [92]:
query_text5

'The plane glided in the air. Its two gigantic wings were reflecting the sunlight while its engine was roaring in the sky.'

In [93]:
r = await scorer.query_openai(query_text5, chunks_retrieved[:])

In [94]:
r

[{'judge0': 2, 'judge1': 2, 'average': 2.0},
 {'judge0': 2, 'judge1': 2, 'average': 2.0},
 {'judge0': 2, 'judge1': 2, 'average': 2.0},
 {'judge0': 0, 'judge1': 0, 'average': 0.0}]

In [55]:
chunks_retrieved[0]

'The passengers in the huge plane high above them gave little thought to what passed below, engrossed with their papers or books, or engaged in casual conversation. This monotonous trip was boring to most of them. It seemed a waste of time to spend six good hours in a short 3,500 mile trip. There was nothing to do, nothing to see, except a slowly passing landscape ten miles below. No details could be distinguished, and the steady low throb of the engines, the whirring of the giant propellers, the muffled roar of the air, as it rushed by, combined to form a soothing lullaby of power. It was all right for pleasure seekers and vacationists, but business men were in a hurry.'

In [56]:
[attr for attr in dir(scorer) if "prompt" in attr.lower()]

['create_queries_unique_prompt', 'prompt1', 'prompt2']

In [63]:
for attr in dir(scorer):
    if not attr.startswith("_"):
        print(attr)

adapt_prompts
average_scores
get_prompts
get_required_columns
init
llm
load_prompts
name
output_type
process_score
required_columns
retry
save_prompts
set_prompts
single_turn_ascore
single_turn_score
template_relevance1
template_relevance2
train


In [65]:
scorer.template_relevance1

'### Instructions\n\nYou are a world class expert designed to evaluate the relevance score of a Context in order to answer the Question.\nYour task is to determine if the Context contains proper information to answer the Question.\nDo not rely on your previous knowledge about the Question.\nUse only what is written in the Context and in the Question.\nFollow the instructions below:\n0. If the context does not contains any relevant information to answer the question, say 0.\n1. If the context partially contains relevant information to answer the question, say 1.\n2. If the context contains any relevant information to answer the question, say 2.\nYou must provide the relevance score of 0, 1, or 2, nothing else.\nDo not explain.\n### Question: {query}\n\n### Context: {context}\n\nDo not try to explain.\nAnalyzing Context and Question, the Relevance score is '

In [64]:
import inspect
print(inspect.getsource(ContextRelevance))

@dataclass
class ContextRelevance(MetricWithLLM, SingleTurnMetric):
    """Parameters:
    Score the relevance of the retrieved contexts be based on the user input.

    Input:
        data: list of Dicts with keys: user_input, retrieved_contexts
    Output:
        0.0: retrieved_contexts is not relevant for the user_input
        0.5: retrieved_contexts is partially relevant for the user_input
        1.0: retrieved_contexts is fully relevant for the user_input
    """

    name: str = field(default="nv_context_relevance", repr=True)  # type: ignore
    _required_columns: t.Dict[MetricType, t.Set[str]] = field(
        default_factory=lambda: {
            MetricType.SINGLE_TURN: {
                "user_input",
                "retrieved_contexts",
            },
        }
    )
    template_relevance1 = (
        "### Instructions\n\n"
        "You are a world class expert designed to evaluate the relevance score of a Context"
        " in order to answer the Question.\n"
        "Y

In [60]:
scorer.__dict__

{'_required_columns': {<MetricType.SINGLE_TURN: 'single_turn'>: {'retrieved_contexts',
   'user_input'}},
 'name': 'nv_context_relevance',
 'llm': InstructorLLM(provider='openai', model='gpt-4o-mini', client=<AsyncInstructor:async>, temperature=0.01, max_tokens=1024, top_p=0.1),
 'output_type': None}

In [23]:
scorer.judge1_prompt

AttributeError: 'ContextRelevance' object has no attribute 'judge1_prompt'

In [24]:
from tqdm import tqdm
async def assess_contexts(df_thresholded, query_text):
    all_results = []
    for p in tqdm(df_thresholded["paragraphs"].values.tolist()): 
        result = await scorer.ascore(
            user_input=query_text,
            retrieved_contexts=p
        )
        all_results.append(result)
        print(p, result)
        
    return all_results

In [25]:
result = await assess_contexts(df_thresholded[:2], query_text5)

NameError: name 'df_thresholded' is not defined

In [119]:
query_text5

'The plane glided in the air. Its two gigantic wings were reflecting the sunlight while its engine was roaring in the sky.'

In [136]:
scorer.judge1_prompt.instruction

'REPLY 2. Return your response as JSON in this format: {"rating": X}'

In [138]:
scorer.judge1_prompt.BasePrompt.to_string()

AttributeError: 'ContextRelevanceJudge1Prompt' object has no attribute 'BasePrompt'

In [121]:
import ragas
print(ragas.__file__)

/proj/berzelius-2025-303/users/x_gabdu/random/ProjectAlbalat/.venv/lib/python3.11/site-packages/ragas/__init__.py


In [22]:
print(scorer.judge1_prompt.to_string(sample))

AttributeError: 'ContextRelevance' object has no attribute 'judge1_prompt'

In [80]:
df_thresholded["paragraphs"].values[-100:-97]

<ArrowStringArray>
['The passengers in the huge plane high above them gave little thought to what passed below, engrossed with their papers or books, or engaged in casual conversation. This monotonous trip was boring to most of them. It seemed a waste of time to spend six good hours in a short 3,500 mile trip. There was nothing to do, nothing to see, except a slowly passing landscape ten miles below. No details could be distinguished, and the steady low throb of the engines, the whirring of the giant propellers, the muffled roar of the air, as it rushed by, combined to form a soothing lullaby of power. It was all right for pleasure seekers and vacationists, but business men were in a hurry.',
                                                                                                                                                                                                                                                                                                         '

In [71]:
print(scorer.single_turn_ascore(sample))

AttributeError: 'ContextRelevance' object has no attribute 'single_turn_ascore'

In [57]:
await scorer.ascore(sample)

AttributeError: 'ContextRelevance' object has no attribute 'ascore'

In [58]:
result = await scorer.ascore(
    user_input="When and where was Albert Einstein born?",
    retrieved_contexts=[
        "Albert Einstein was born March 14, 1879.",
        "Albert Einstein was born at Ulm, in Württemberg, Germany.",
    ]
)

AttributeError: 'ContextRelevance' object has no attribute 'ascore'

In [59]:
ragas.__version__

'0.4.1'

In [61]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextRelevance

# Setup LLM
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Create metric
scorer = ContextRelevance(llm=llm)

# Evaluate
result = await scorer.ascore(
    user_input="When and Where Albert Einstein was born?",
    retrieved_contexts=[
        "Albert Einstein was born March 14, 1879.",
        "Albert Einstein was born at Ulm, in Württemberg, Germany.",
    ]
)
print(f"Context Relevance Score: {result.value}")

Context Relevance Score: 1.0


In [30]:
def exp(N = 100000):
    us = np.random.uniform(size=(N, 3))*2*np.pi
    x1 = np.concatenate([np.cos(us[:, 0, None]), np.sin(us[:, 0, None])], axis=-1)
    x2 = np.concatenate([np.cos(us[:, 1, None]), np.sin(us[:, 1, None])], axis=-1)
    x3 = np.concatenate([np.cos(us[:, 2, None]), np.sin(us[:, 2, None])], axis=-1)
    dot12 = (x1*x2).sum(-1)
    dot13 = (x1*x3).sum(-1)
    dot23 = (x2*x3).sum(-1)
    return (dot12 < 0) | (dot23 < 0) | (dot13 < 0) 
    
    

In [31]:
np.mean(exp())

np.float64(0.81234)

In [32]:
3/8

0.375